# Inferencja modelu do klasyfikacji utworów rapowych


## Import bibliotek i patchowanie


In [1]:
import os

import numpy as np
import pandas as pd
import torch
from peft import PeftModel

os.environ["UNSLOTH_DISABLE_FAST_GENERATION"] = "1"

# Unsloth przed transformers (optymalizacje + patch Gemma4)
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

from transformers import (
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
)
from transformers.configuration_utils import PretrainedConfig
from transformers.modeling_layers import GenericForSequenceClassification
from transformers.models.auto.modeling_auto import (
    MODEL_FOR_SEQUENCE_CLASSIFICATION_MAPPING,
)
from transformers.models.gemma4.configuration_gemma4 import Gemma4Config
from transformers.models.gemma4.modeling_gemma4 import Gemma4PreTrainedModel

# Workaround: patch Unsloth ukrywa num_kv_shared_layers==0 i psuje AutoConfig.__repr__
_orig_to_diff_dict = PretrainedConfig.to_diff_dict

def _safe_to_diff_dict(self):
    if "Gemma4" in self.__class__.__name__:
        return self.to_dict()
    return _orig_to_diff_dict(self)


PretrainedConfig.to_diff_dict = _safe_to_diff_dict

# Transformers 5.10 nie ma jeszcze gotowej klasy dla Gemma 4 - rejestrujemy ją ręcznie.
# Gemma4Config jest composite: hidden_size jest w text_config, nie na top-level config.
class Gemma4ForSequenceClassification(GenericForSequenceClassification, Gemma4PreTrainedModel):
    def __init__(self, config):
        Gemma4PreTrainedModel.__init__(self, config)
        self.num_labels = config.num_labels
        setattr(self, self.base_model_prefix, AutoModel.from_config(config))
        self.score = torch.nn.Linear(
            config.get_text_config().hidden_size, self.num_labels, bias=False
        )
        self.post_init()


MODEL_FOR_SEQUENCE_CLASSIFICATION_MAPPING.register(
    Gemma4Config, Gemma4ForSequenceClassification, exist_ok=True
)

W0614 23:28:50.053000 13512 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\Damia\Desktop\Projekt-PJN\.venv\Lib\site-packages\unsloth\__init__.py:165: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
[ERROR] `loss` is part of Qwen3VLMoeCausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in c:\Users\Damia\Desktop\Projekt-PJN\.venv\Lib\site-packages\transformers\models\qwen3_vl_moe\modeling_qwen3_vl_moe.py.
[ERROR] `logits` is part of Qwen3VLMoeCausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in c:\Users\Damia\Desktop\Projekt-PJN\.venv\Lib\site-packages\transformers\models\qwen3_vl_moe\modeling_qwen3_vl_moe.py.
[ERROR] `loss` is part of Qwen3_5MoeCausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in c:\Users\Damia\Desktop\Projekt-PJN\.venv\Lib\site-packages\transformers\models\qwen3_5_moe\modeling_qwen3_5_moe.py.
[ERROR] `logits` is part of Qwen3_5MoeCausalLMOutputWithPast.__init__'s signature, but not documented.

## Wczytanie danych i etykiet


In [2]:
DATASET_DIR = "dataset"

# Etykiety odtwarzamy z train.json, a zwrotki do inferencji bierzemy ze zbioru testowego.
train_df = pd.read_json(os.path.join(DATASET_DIR, "train.json"))
test_df = pd.read_json(os.path.join(DATASET_DIR, "test.json"))

labels = sorted(train_df["label"].unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
NUM_LABELS = len(labels)

# Zostawiamy tylko zwrotki testowe o etykietach znanych modelowi.
test_df = test_df[test_df["label"].isin(label2id)].reset_index(drop=True)

print(f"Liczba klas: {NUM_LABELS}")
print(f"Zbiór testowy: {len(test_df)} zwrotek")
test_df.head(3)

Liczba klas: 20
Zbiór testowy: 1203 zwrotek


,song,section,text,label
0,Marcelo Burlon,refren,Siedzę w Mediolanie jak Marcelo Burlon\nŻycie ...,Żabson
1,All Black,zwrotka,"Zmieniasz się całe życie, umrzesz taki sam, ta...",Quebonafide
2,Hood Angel,refren,"Zarabiam, oh, zarabiam big money, oh\nNie meen...",Żabson


## Wczytanie modelu bazowego i adapterów LoRA

Model bazowy `unsloth/gemma-4-E2B` ładujemy w bf16 LoRA na GPU, a na sprzęcie bez GPU na CPU bez kwantyzacji.
Następnie nakładamy zapisane adaptery LoRA i wytrenowaną głowicę klasyfikacyjną z katalogu `gemma4-e2b-polish-rap-classifier` przez `PeftModel.from_pretrained`.


In [3]:
MODEL_NAME = "unsloth/gemma-4-E2B"
ADAPTER_DIR = "gemma4-e2b-polish-rap-classifier"
MAX_LENGTH = 650  # ok. 99. percentyl

has_cuda = torch.cuda.is_available()
print(f"Urządzenie: {'GPU (bf16 LoRA)' if has_cuda else 'CPU (bez kwantyzacji)'}")

# Wczytanie modelu bazowego
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_LENGTH,
    load_in_4bit=False,
    load_in_16bit=has_cuda,
    dtype=None if has_cuda else "float32",
    auto_model=AutoModelForSequenceClassification,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    trust_remote_code=False,
    ignore_mismatched_sizes=True,
    use_exact_model_name=True,
    device_map="auto",
)

# Nałożenie wytrenowanych adapterów LoRA i głowicy klasyfikacyjnej.
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model.eval()

# Tokenizer wczytujemy z katalogu z adapterami.
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
if not tokenizer.chat_template:
    tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Wczytano model i adaptery z: {ADAPTER_DIR}")

Urządzenie: GPU (bf16 LoRA)
==((====))==  Unsloth 2026.6.1: Fast Gemma4 patching. Transformers: 5.10.2.
   \\   /|    NVIDIA GeForce RTX 5060 Ti. Num GPUs = 1. Max memory: 15.928 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Gemma4ForSequenceClassification LOAD REPORT from: unsloth/gemma-4-E2B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Wczytano model i adaptery z: gemma4-e2b-polish-rap-classifier


## Formatowanie wejścia i funkcja predykcji

Używamy dokładnie tego samego promptu i formatu wejścia co podczas treningu.


In [4]:
CLASSIFICATION_PROMPT = """Przeczytaj poniższą zwrotkę polskiego utworu rapowego i określ, kto jest wykonawcą.

Tekst zwrotki:
{verse}"""


def format_input(verse: str) -> str:
    messages = [{"role": "user", "content": CLASSIFICATION_PROMPT.format(verse=verse)}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return text.removeprefix("<bos>")


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


@torch.no_grad()
def predict_artist(verse: str, top_k: int = 5):
    inputs = tokenizer(
        format_input(verse),
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(DEVICE)
    probs = model(**inputs).logits.softmax(-1)[0]
    top = probs.topk(min(top_k, NUM_LABELS))
    return [(id2label[i.item()], p.item()) for p, i in zip(top.values, top.indices)]

## Inferencja dla wybranej zwrotki ze zbioru testowego


In [5]:
SAMPLE_INDEX = 0  # Indeks zwrotki ze zbioru testowego

row = test_df.iloc[SAMPLE_INDEX]
verse = row["text"]
true_label = row["label"]

ranking = predict_artist(verse, top_k=5)
top_label = ranking[0][0]
status = "TRAFIONO" if top_label == true_label else "POMYŁKA"

print(f"Utwór: {row.get('song', '—')}")
print(f"Fragment: {' / '.join(verse.splitlines()[:3])}")
print(f"Prawdziwy: {true_label}")
print(f"[{status}] Predykcja (top-5):")
for artist, prob in ranking:
    marker = " <-- prawda" if artist == true_label else ""
    print(f"  {artist:30s} {prob:.1%}{marker}")

`use_return_dict` is deprecated! Use `return_dict` instead!


Utwór: Marcelo Burlon
Fragment: Siedzę w Mediolanie jak Marcelo Burlon / Życie nie jest tanie jak Marcelo Burlon / Wiesz że wniosłem ten styl i dlatego mówią
Prawdziwy: Żabson
[TRAFIONO] Predykcja (top-5):
  Żabson                         81.2% <-- prawda
  Louis Villain                  7.1%
  OG Olgierd                     2.7%
  Guzior                         1.5%
  Oki                            1.3%


## Inferencja dla losowej zwrotki ze zbioru testowego


In [6]:
row = test_df.sample(1).iloc[0]
verse = row["text"]
true_label = row["label"]

ranking = predict_artist(verse, top_k=5)
top_label = ranking[0][0]
status = "TRAFIONO" if top_label == true_label else "POMYŁKA"

print(f"Utwór: {row.get('song', '—')}")
print(f"Fragment: {' / '.join(verse.splitlines()[:3])}")
print(f"Prawdziwy: {true_label}")
print(f"[{status}] Predykcja (top-5):")
for artist, prob in ranking:
    marker = " <-- prawda" if artist == true_label else ""
    print(f"  {artist:30s} {prob:.1%}{marker}")

Utwór: Highaf
Fragment: Pytają czemu nie wydaję nowych numerów stale / Pytają czemu nie nagrywam nowych bangerów stale / Pytają czemu nie dogram typkowi refrenu, frajer!
Prawdziwy: Bedoes
[POMYŁKA] Predykcja (top-5):
  Guzior                         42.0%
  Louis Villain                  14.7%
  OG Olgierd                     8.0%
  Oki                            7.2%
  Bedoes                         6.1% <-- prawda
